In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from typing import Any, Literal
import sys
import json

from openai import OpenAI
import pydantic

from icecream import ic
from tqdm import tqdm

load_dotenv()

MODEL_NAME = "deepseek-v4-pro"
BASE_URL = "https://api.deepseek.com"
TEMPERATURE = 1.0
REASONING=True
REASONING_EFFORT = "medium"
MAX_TOKENS = (8192, 10000)[REASONING]
TIMEOUT_SECONDS = 240.0
PAIRS_PER_PROMPT = 3
REGENERATION_ATTEMPTS = 3

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "prompts").is_dir() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROMPTS_ROOT = PROJECT_ROOT / "prompts"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "generated"
SELECTED_PROMPTS = [
    Path("general_validation.md"),
    # Path("metrics/size_compliance.md"),
    Path("metrics/intrachunk_cohesion.md"),
    Path("metrics/contextual_coherence.md"),
    Path("metrics/boundary_clarity.md"),
    Path("metrics/chunk_score.md"),
    Path("metrics/hope_concept_unity.md"),
    Path("metrics/hope_semantic_independence.md"),
    Path("metrics/hope_information_preservation.md"),
]

client = OpenAI(
    api_key=os.environ["API_KEY"],
    base_url=BASE_URL,
    timeout=TIMEOUT_SECONDS,
)


In [ ]:
class ChunkingVariant(pydantic.BaseModel):
    chunks: list[str]
    rationale: str
    focus: dict[str, Any] = pydantic.Field(default_factory=dict)


class SyntheticChunkingExample(pydantic.BaseModel):
    document_title: str
    source_document: str

    positive: ChunkingVariant
    negative: ChunkingVariant

    controlled_change: str

    expected_relation: Literal[
        "positive_higher_than_negative"
    ]

SyntheticChunkingExample.model_json_schema()

In [ ]:
def save_json(
    data: dict[str, Any] | list[dict[str, Any]], path: Path
) -> Path:
    """Save JSON objects in a human-readable UTF-8 file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return path

In [ ]:
def generate(
    system_prompt,
    user_prompt,
):
    result = None
    for attempt in range(REGENERATION_ATTEMPTS):
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            response_format={"type": "json_object"},
            extra_body={"thinking": {"type": ("disabled", "enabled")[REASONING]}},
            reasoning_effort=REASONING_EFFORT,
        )
        content = response.choices[0].message.content
        try:
            # ic(response)
            # ic(content)
            result = SyntheticChunkingExample.model_validate_json(content).model_dump()
            break
        except pydantic.ValidationError as e:
            tqdm.write("Retrying..")
    return result
    

In [ ]:
# def llm_judge(content):
    

In [ ]:
system_prompt = (PROMPTS_ROOT / "system.md").read_text(encoding="utf-8")

for prompt_path in tqdm(SELECTED_PROMPTS, desc="Prompts", position=0):
    prompt_name = prompt_path.stem
    user_prompt = (PROMPTS_ROOT / prompt_path).read_text(encoding="utf-8")
    results = []
    output_path = ""
    for pair_number in tqdm(range(1, PAIRS_PER_PROMPT + 1), desc="Items", position=1, leave=False):
        result = generate(
            system_prompt=system_prompt,
            user_prompt=user_prompt
        )
        
        results.append(result)

        output_path = save_json(
            results, OUTPUT_ROOT / f"{prompt_name}.json"
        )
    tqdm.write(f"Saved {output_path}")